In [1]:
import neo4j
from neo4j_graphrag.llm import OpenAILLM as LLM
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings as Embeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation.graphrag import GraphRAG

In [2]:
import asyncio
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import AzureOpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import AzureOpenAILLM

NEO4J_URI = "neo4j://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "neo4j_embedding"

In [3]:
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [4]:
neo4j_driver.verify_connectivity()

In [20]:
from config import azure_config

llm = AzureOpenAILLM(
    model_name=azure_config.model_name,
    model_params={"temperature": 0},
    api_key=azure_config.api_key,
    azure_endpoint=azure_config.endpoint,
    api_version=azure_config.api_version,
    azure_deployment="gpt-4o"    
)

embedder = AzureOpenAIEmbeddings(
    model=azure_config.embedding_model_name,
    api_key=azure_config.api_key,
    azure_endpoint=azure_config.endpoint,
    api_version=azure_config.api_version,
    # azure_deployment="text-embedding-3-small"
)

In [22]:
len(embedder.embed_query(llm.invoke("What is the capital of France?").content))

1536

In [28]:
kg_builder_pdf = SimpleKGPipeline(
   llm=llm,
   driver=neo4j_driver,
   embedder=embedder,
   from_pdf=False,
)

In [35]:
from markitdown import MarkItDown

md = MarkItDown()
pdf_content = md.convert("Graph_Databases_for_Beginners.pdf").text_content
# print(result.text_content)

In [ ]:
relations = [
    'USES',       # For instance, a technology that uses a specific subject
    'LOCATED_IN', # For example, a country that is located in a specific location
    'CONNECTED_TO', # To describe a relationship between two technologies or subjects
    'DEVELOPS'    # To indicate a country that develops a specific technology
]
entities = ['SUBJECT', 'TECHNOLOGY', 'COUNTRY', 'LOCATION']
kg_builder_pdf = SimpleKGPipeline(
    llm=llm,
    driver=neo4j_driver,
    embedder=embedder,
    entities=entities,
    relations=relations,
    on_error="IGNORE",
    from_pdf=True,
)
# asyncio.run(kg_builder.run_async(text=text))
# await kg_builder_pdf.run_async(text=pdf_content,)
await kg_builder_pdf.run_async(file_path="Graph_Databases_for_Beginners.pdf",)

PipelineResult(run_id='c631c5af-81ca-424b-b320-7d109f238356', result={'resolver': {'number_of_nodes_to_resolve': 265, 'number_of_created_nodes': 190}})

In [ ]:
from neo4j_graphrag.indexes import create_vector_index
INDEX_NAME = "vector-index-name"

# Create the index
create_vector_index(
    neo4j_driver,
    INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [70]:
# Initialize the retriever
retriever = VectorRetriever(neo4j_driver, INDEX_NAME, embedder)

# Instantiate the RAG pipeline
rag = GraphRAG(retriever=retriever, llm=llm)

# Query the graph
# query_text = "Which other companies are related to eBay?"
# query_text = "Who is Duke Leto Atreides? tell me everything about him."
# query_text = "Who works with Bob?"
query_text = "Who is the manager of Bob?"
# query_text = "What is the relationship between Bob and Edward?"
response = rag.search(query_text=query_text, retriever_config={"top_k": 10}, return_context=True)
print(response.answer)

/opt/miniconda3/envs/lib/python3.12/site-packages/neo4j_graphrag/retrievers/vector.py:200: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


Alice is the manager of Bob.

In [58]:
from rich import print
print(response)

RagResultModel(
    answer='Alice, Charlie, Davina, and Edward know Bob.',
    retriever_result=RetrieverResult(
        items=[
            RetrieverResultItem(
                content="{'id': '8233a9aa-dd36-49ea-b841-21d88b7f4289', 'index': 6, 'text': 'this initial data 
modeling attempt looks like an accurate representation of Bob’s \\nemail activity; after all, we can easily see 
that Bob (an alias of Alice) emailed Charlie while BCC’ing \\nEdward and CC’ing Davina. But we can’t see the most 
important part of all: the email itself.\\nA beginning data modeler might try to remedy the situation by adding 
properties to the \\nEMAILED relationship, representing the email’s attributes as properties. However, that’s not a
\\nlong-term solution. Even with properties attached to each EMAILED relationship, we wouldn’t \\nbe able to 
correlate connections between EMAILED, CC and BCC relationships – and those \\ncorrelating relationships are 
exactly what we need for our fraud detection solution.\\nThis is the perfect example of a common data modeling 
mistake. In everyday English, it’s easy \\nand convenient to shorten the phrase “Bob sent an email to Charlie” to 
“Bob emailed Charlie.” \\nThis shortcut made us focus on the verb “emailed” rather than the email as an object 
itself. \\nAs a result, our incomplete model keeps us from the insights we’re looking for.\\nThe Fix: A Stronger 
Fraud Detection Data Model\\nTo fix our weak model, we need to add nodes to our graph model that represent each of 
the \\nemails exchanged. Then, we need to add new relationships to track who wrote the email and \\nto whom it was 
sent, CC’ed and BCC’ed.\\nThe result is another star-shaped graph, but this time the email is at the center, 
allowing us \\nto efficiently track its relationship to Bob and possibly some suspicious behavior.\\nFIGURE 4.2: 
Our second attempt at a fraud \\ndetection data model. This iteration allows us \\nto more easily trace the 
relationships of who is \\nsending and receiving each email 
message.\\nusername:\\nAlice\\nUser\\nusername:\\nCharlie\\nUser\\nusername:\\nDavina\\nUser\\nusername:\\nBob\\nUs
er\\nusername:\\nEdward\\nUser\\nHi Charlie,...\\nKind 
regards,\\nBob\\nEmail\\nSENT\\nTO\\nCC\\nCC\\nBCC\\nneo4j.com\\n11\\nGraph Databases For Beginners\\nOf course we 
aren’t interested in tracking just one email but many, each with its own web of interactions to explore. Over time,
our \\nemail server logs more interactions, giving us something like the graph below.\\nEmail\\nid: 
1\\ncontent:\\nemail\\ncontents\\nid: 2\\ncontent:\\nemail\\ncontents\\nid: 5\\ncontent:\\nemail\\ncontents\\nid: 
3\\ncontent:\\nemail\\ncontents\\nEmail\\nid: 
4\\ncontent:\\nemail\\ncontents\\nusername:\\nAlice\\nUser\\nusername:\\nEdward\\nUser\\nusername:\\nCharlie\\nUser
\\nusername:\\nDavina\\nUser\\nusername:\\nBob\\nUser\\nEmail EmailEmail\\nTO\\nTO\\nTO\\nSENTSENT\\nSENT\\nBCC 
BCC\\nBCC\\nSENT\\nALIAS_OF\\nSENT\\nTO\\nCC\\nTO\\nCC\\nCC\\nBCC\\nTO\\nTO\\nFIGURE 4.3: A data model showing many
\\nemails over time and their various \\nrelationships, including the sender and \\nthe direct, CC and BCC 
receivers.\\nThe Next Step: Tracking Email Replies\\nAt this point, our data model is more robust, but it isn’t 
complete. We can see who sent and received emails, and we can see the \\ncontent of the emails themselves. 
Nevertheless, we can’t track any replies or forwards of our given email communications. In the case \\nof fraud or 
cybersecurity, we need to know if critical business information has been leaked or compromised.\\nTo complete this 
upgrade, beginners might be tempted to simply add FORWARDED and REPLIED_TO relationships to our graph \\nmodel, 
like in the example below.\\nusername:\\nAlice\\nusername:\\nCharlie\\nUser User\\nEmail 
id:\\n1234\\nEmail\\nusername:\\nDavina\\nUser\\nREPLIED_TO TO\\nFORWARDED\\nFIGURE 4.4: Our updated data model 
with \\nFORWARDED and REPLIED_TO relationships in \\naddition to the original TO 

In [38]:
# List the entities and relations the LLM should look for in the text
entities = ["Person", "House", "Planet"]
relations = ["PARENT_OF", "HEIR_OF", "RULES"]
potential_schema = [
    ("Person", "PARENT_OF", "Person"),
    ("Person", "HEIR_OF", "House"),
    ("House", "RULES", "Planet"),
]

# Instantiate the SimpleKGPipeline
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=neo4j_driver,
    embedder=embedder,
    entities=entities,
    relations=relations,
    on_error="IGNORE",
    from_pdf=False,
)

# Run the pipeline on a piece of text
text = (
    "The son of Duke Leto Atreides and the Lady Jessica, Paul is the heir of House "
    "Atreides, an aristocratic family that rules the planet Caladan."
)
asyncio.run(kg_builder.run_async(text=text))

PipelineResult(run_id='3e6860c5-b775-440e-9523-0951776a7b75', result={'resolver': {'number_of_nodes_to_resolve': 5, 'number_of_created_nodes': 5}})

In [39]:
from neo4j_graphrag.indexes import create_vector_index
INDEX_NAME = "vector-index-name"

# Create the index
create_vector_index(
    neo4j_driver,
    INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [40]:
# Generate an embedding for some text
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
text = (
    "The son of Duke Leto Atreides and the Lady Jessica, Paul is the heir of House "
    "Atreides, an aristocratic family that rules the planet Caladan."
)
vector = embedder.embed_query(text)

# Upsert the vector
upsert_vectors(
    neo4j_driver,
    ids=["1234"],
    embedding_property="vectorProperty",
    embeddings=[vector],
    entity_type=EntityType.NODE,
)

In [42]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import VectorRetriever

# Initialize the retriever
retriever = VectorRetriever(neo4j_driver, INDEX_NAME, embedder)

# Instantiate the RAG pipeline
rag = GraphRAG(retriever=retriever, llm=llm)

# Query the graph
query_text = "Who is Paul Atreides?"
response = rag.search(query_text=query_text, retriever_config={"top_k": 5})
print(response.answer)

/opt/miniconda3/envs/lib/python3.12/site-packages/neo4j_graphrag/generation/graphrag.py:117: DeprecationWarning: The default value of 'return_context' will change from 'False' to 'True' in a future version.
  warnings.warn(
/opt/miniconda3/envs/lib/python3.12/site-packages/neo4j_graphrag/retrievers/vector.py:200: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


Paul Atreides is the son of Duke Leto Atreides and the Lady Jessica, and he is the heir of House Atreides, an aristocratic family that rules the planet Caladan.


In [ ]:
# 1. Build KG and Store in Neo4j Database

# 2. KG Retriever
vector_retriever = VectorRetriever(
   neo4j_driver,
   index_name="text_embeddings",
   embedder=embedder
)

# 3. GraphRAG Class
# llm = LLM(model_name="gpt-4o")
rag = GraphRAG(llm=llm, retriever=vector_retriever)

# 4. Run
response = rag.search( "List benefits of graph databases")
print(response.answer)

TypeError: 'id' as a property name is not allowed